In [1]:
# Cài đặt các thư viện lõi
!pip install pyspark==3.5.0 faiss-cpu snowflake-connector-python[pandas] -q

print(" Đã cài đặt xong PySpark, FAISS và Snowflake Connector!")

 Đã cài đặt xong PySpark, FAISS và Snowflake Connector!


In [2]:
import snowflake.connector
import pandas as pd
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName('SmartMoney_ReRanking') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()
spark.sparkContext.setLogLevel('WARN')

print("1. Đang kết nối Snowflake...")
conn = snowflake.connector.connect(
    user      = 'MANHDAN',
    password  = '1', # Lưu ý: Cân nhắc dùng biến Secrets của Colab trong thực tế
    account   = 'ms14138.ap-southeast-1',
    database  = 'BIGDATA_DB',
    schema    = 'STAGING',
    autocommit= True,
)

# 2. Truy vấn toàn bộ bảng chứa Vector + PageRank
query = "SELECT * FROM BIGDATA_DB.STAGING.LIGHTGCN_USER_FACTORS"
print("2. Đang kéo dữ liệu về Pandas...")
cursor = conn.cursor()
cursor.execute(query)
pdf_users = cursor.fetch_pandas_all()
cursor.close()
conn.close()

# 3. Chuẩn hóa tên cột
pdf_users.columns = [col.lower() for col in pdf_users.columns]
col_names = pdf_users.columns.tolist()

# Ép tên cột đầu tiên thành 'id' và cột cuối cùng thành 'pagerank_score'
pdf_users.rename(columns={col_names[0]: 'id', col_names[-1]: 'pagerank_score'}, inplace=True)

# Chuyển qua Spark DataFrame để chuẩn bị cho phần sau
df_users = spark.createDataFrame(pdf_users)

print(f" Tải thành công {len(pdf_users):,} ví!")
print(f" Cấu trúc: 1 cột ID, {len(col_names)-2} cột Features, 1 cột PageRank")
df_users.select('id', col_names[1], col_names[2], 'pagerank_score').show(5)

1. Đang kết nối Snowflake...
2. Đang kéo dữ liệu về Pandas...
 Tải thành công 464,423 ví!
 Cấu trúc: 1 cột ID, 128 cột Features, 1 cột PageRank
+-------+------------+------------+--------------------+
|     id|          f0|          f1|      pagerank_score|
+-------+------------+------------+--------------------+
|9951813|   0.0321244| 0.019822285|5.362883300929021E-8|
|9953261| -0.01656527|-0.049694654|2.600595111513314...|
|9953321|0.0059171333|-0.044246897|1.302678226768769...|
|9953615| -0.02187384|-0.010182623|2.186657192721623...|
|9953906| -0.03512123|-0.021911053|8.011365997454179E-8|
+-------+------------+------------+--------------------+
only showing top 5 rows



In [3]:
import numpy as np
import faiss
import time

print("--- BẮT ĐẦU CHẠY FAISS (TÌM TOP 100 HÀNG XÓM) ---")
start_time = time.time()

# 1. Bóc tách riêng 128 cột Vector (Bỏ cột 'id' và 'pagerank_score')
feature_cols = [c for c in pdf_users.columns if c not in ['id', 'pagerank_score']]

# 2. Chuyển đổi sang định dạng Float32 cho FAISS
vectors = pdf_users[feature_cols].values.astype('float32')
user_ids = pdf_users['id'].values
dimension = vectors.shape[1]

print(f"Đang huấn luyện bộ phân cụm trên {len(vectors):,} ví ({dimension} chiều)...")
nlist = 1000
quantizer = faiss.IndexFlatL2(dimension)
index_ivf = faiss.IndexIVFFlat(quantizer, dimension, nlist, faiss.METRIC_L2)
index_ivf.train(vectors)
index_ivf.add(vectors)

# 3. Quét Top 100
TOP_K = 100
index_ivf.nprobe = 20
print(f"Đang quét tìm hàng xóm...")
distances, indices = index_ivf.search(vectors, TOP_K + 1)

print(f" FAISS HOÀN TẤT! Thời gian chạy: {round(time.time() - start_time, 2)} giây")

--- BẮT ĐẦU CHẠY FAISS (TÌM TOP 100 HÀNG XÓM) ---
Đang huấn luyện bộ phân cụm trên 464,423 ví (128 chiều)...
Đang quét tìm hàng xóm...
 FAISS HOÀN TẤT! Thời gian chạy: 300.95 giây


In [4]:
import pandas as pd
from snowflake.connector.pandas_tools import write_pandas

print("--- BẮT ĐẦU RE-RANKING VÀ ĐỔI FORMAT BẢNG ---")

# 1.Ép kiểu dữ liệu triệt để cho Dictionary
# Bắt buộc key phải là số nguyên (int) và value là số thực (float)
keys = pdf_users['id'].astype(int).values
vals = pdf_users['pagerank_score'].astype(float).values
pagerank_dict = dict(zip(keys, vals))

# 2. Xử lý logic lọc Top 5 và "Trải phẳng" (Flatten) dữ liệu
print("Đang chấm điểm, lọc Top 5 và tạo bảng phẳng...")
final_results = []

for i in range(len(user_ids)):
    target_wallet = int(user_ids[i])

    # Lấy 100 hàng xóm từ FAISS (bỏ qua vị trí 0 là chính nó)
    neighbors = indices[i][1:]
    dists = distances[i][1:]

    candidates = []
    for j in range(len(neighbors)):
        n_id = int(user_ids[neighbors[j]])
        d = float(dists[j])

        # Tra cứu PageRank, nếu khác kiểu dữ liệu hoặc không có sẽ an toàn trả về 0.0
        pr = float(pagerank_dict.get(n_id, 0.0))
        candidates.append((n_id, pr, d))

    # Sắp xếp: Ưu tiên 1 là PageRank cao nhất (desc), Ưu tiên 2 là Khoảng cách nhỏ nhất (asc)
    candidates.sort(key=lambda x: (-x[1], x[2]))

    # Cắt lấy chính xác Top 5
    top_5 = candidates[:5]

    # Tạo 5 dòng riêng biệt cho mỗi ví mục tiêu
    for rank, c in enumerate(top_5, start=1):
        final_results.append({
            "USER_ID": target_wallet,
            "USER_ID_SIMILARITY": c[0],
            "RANK": rank,
            "PAGERANK": c[1]
        })

# 3. Tạo Pandas DataFrame
pdf_final_mentors = pd.DataFrame(final_results)
print(f" Đã tạo xong bảng! Tổng số dòng chuẩn bị ghi: {len(pdf_final_mentors):,} dòng")

# 4. Ghi ngược lại (Write-back) lên Snowflake
print("Đang đẩy dữ liệu lên Snowflake...")
try:
    conn_write = snowflake.connector.connect(
        user      = 'MANHDAN',
        password  = '1',
        account   = 'ms14138.ap-southeast-1',
        database  = 'BIGDATA_DB',
        schema    = 'STAGING',
        autocommit= True,
    )


    TABLE_NAME = "RECOMMENDATION_TOP5_MENTORS"
    success, num_chunks, num_rows, output = write_pandas(
        conn=conn_write,
        df=pdf_final_mentors,
        table_name=TABLE_NAME,
        auto_create_table=True, # Tự động tạo bảng nếu chưa có
        overwrite=True          # Xóa bảng cũ và ghi mới
    )

    conn_write.close()

    if success:
        print(f" THÀNH CÔNG ! Đã ghi {num_rows:,} dòng vào bảng {TABLE_NAME}.")
    else:
        print("Có lỗi nhỏ trong quá trình ghi, hãy kiểm tra lại output.")

except Exception as e:
    print(f" Xảy ra lỗi khi ghi lên Snowflake: {e}")

# Xem trước kết quả để kiểm chứng
print("\nBản xem trước dữ liệu :")
print(pdf_final_mentors.head(10))

--- BẮT ĐẦU RE-RANKING VÀ ĐỔI FORMAT BẢNG ---
Đang chấm điểm, lọc Top 5 và tạo bảng phẳng...
 Đã tạo xong bảng! Tổng số dòng chuẩn bị ghi: 2,322,115 dòng
Đang đẩy dữ liệu lên Snowflake...
 THÀNH CÔNG ! Đã ghi 2,322,115 dòng vào bảng RECOMMENDATION_TOP5_MENTORS.

Bản xem trước dữ liệu :
   USER_ID  USER_ID_SIMILARITY  RANK      PAGERANK
0  9951813            21407323     1  4.401767e-07
1  9951813            20482907     2  3.030710e-07
2  9951813            17245395     3  2.910488e-07
3  9951813            13086760     4  2.716005e-07
4  9951813            12918038     5  1.537018e-07
5  9953261            18394919     1  4.047363e-07
6  9953261            21092083     2  1.818895e-07
7  9953261            23564250     3  1.671385e-07
8  9953261            13501734     4  1.596031e-07
9  9953261            16421832     5  1.569241e-07
